# Stage 4 · All-cell L1 clustering

Final all-cell integration using the refined anchors (`dist_pc` < p90) → **35 major types**; then split cells into per-major-type `L2/{ct}/` directories.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys

ENTEX_ROOT = os.environ.get("ENTEX_ROOT", "/large_storage/zhoulab/zhoujt/project/ENTEx")
REF_ROOT = os.environ.get("REF_ROOT", "/large_storage/zhoulab/ref")
BOOK_ROOT = os.environ.get("BOOK_ROOT", f"{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas")
sys.path.insert(0, BOOK_ROOT)
import repro_guard

## 4a · mCG all-cell (refined anchors)

_Source: `clustering/merged/L1/01.5kCG_clustering.ipynb`_

In [2]:
from ALLCools.clustering import ConsensusClustering, significant_pc_test, tsne
from ALLCools.plot import categorical_scatter, continuous_scatter
import os
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from ALLCools.integration.seurat_class import SeuratIntegration
from sklearn.metrics import adjusted_rand_score as ARI
from sklearn.preprocessing import normalize

In [3]:
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

In [4]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f"{name}_{i}"] = adata.obsm[f"X_{name}"][:, i]
    return adata

In [5]:
indir = f"{ENTEX_ROOT}/clustering/merged/"
outdir = f"{ENTEX_ROOT}/clustering/"

In [6]:
tmp = anndata.read_h5ad("../L1pre/5kCG_lsi.h5ad")
tmp

In [7]:
mcad = anndata.AnnData(
    obs=tmp.obs[
        [
            "FinalmCReads",
            "mCHFrac",
            "mCGFrac",
            "CisLongContact",
            "Cis/Trans",
            "Donor",
            "Tissue",
            "celltype",
            "ClusterTissue",
        ]
    ]
)
mcad.obsm["5kCG_lsi"] = tmp.obsm["5kCG_pca"].copy()
mcad.obsm["5kCG_u50_tsne"] = tmp.obsm["5kCG_u50_tsne"].copy()

In [8]:
npc = significant_pc_test(mcad, p_cutoff=0.01, obsm="5kCG_lsi", update=False)

In [9]:
npc = 50
mcad.obsm["X_lsi"] = normalize(mcad.obsm["5kCG_lsi"][:, :npc], axis=1)

In [10]:
sample_list = np.sort(mcad.obs["Donor"].astype(str).unique())
sample_list

In [11]:
adata_list = [mcad[mcad.obs["Donor"] == xx] for xx in sample_list]

In [12]:
anchor_index = {}
for xx, adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index("cell")["index"].to_dict()

In [13]:
ct_list = np.sort(
    [xx.split("/")[-2] for xx in glob(f"{outdir}tissue/L2/*-*/mCG_5kb_seurat_anchor_*.hdf")]
)
print(len(ct_list))

In [14]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f"{outdir}tissue/L2/{ct}/mCG_5kb_seurat_anchor_*.hdf")[0]
    adata_file = glob(f"{outdir}tissue/L2/{ct}/mCG_5kb_lsi.h5ad")[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs["Donor"].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs["Donor"] == xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp["dist_pc"], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp["dist_pc"] < thres, ["x1", "x2", "score"]].copy()
    anchor_tmp["x1"] = cell_list[0][anchor_tmp["x1"]].map(anchor_index[sample_tmp[0]])
    anchor_tmp["x2"] = cell_list[1][anchor_tmp["x2"]].map(anchor_index[sample_tmp[1]])
    anchor_tmp["x1_donor"] = sample_tmp[0]
    anchor_tmp["x2_donor"] = sample_tmp[1]
    anchor.append(anchor_tmp)

anchor = pd.concat(anchor, axis=0)

In [15]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i, j]]
        selc = (anchor["x1_donor"] == x1) & (anchor["x2_donor"] == x2)
        anchor_df[(i, j)] = anchor.loc[selc, ["x1", "x2", "score"]]
        if selc.sum() == 0:
            selc = (anchor["x1_donor"] == x2) & (anchor["x2_donor"] == x1)
            anchor_df[(i, j)] = anchor.rename({"x1": "x2", "x2": "x1"}, axis=1).loc[
                selc, ["x1", "x2", "score"]
            ]

In [16]:
integrator = SeuratIntegration()

In [17]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = "X_lsi"
integrator._calculate_local_knn()

integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df

In [18]:
corrected = integrator.integrate(
    key_correct="5kCG_lsi", row_normalize=True, n_components=npc, k_weight=100, sd=1
)

In [19]:
corrected = pd.DataFrame(
    normalize(np.concatenate(corrected, axis=0), axis=1),
    index=np.concatenate([xx.obs.index for xx in adata_list]),
)

mcad.obsm[f"5kCG_u{npc}_seuratL2"] = corrected.loc[mcad.obs.index].values
mcad.obsm[f"5kCG_u{npc}_seuratL2"] = normalize(mcad.obsm[f"5kCG_u{npc}_seuratL2"][:, :npc], axis=1)

tsne(
    mcad,
    obsm=f"5kCG_u{npc}_seuratL2",
    metric="euclidean",
    exaggeration=-1,
    perplexity=50,
    n_jobs=-1,
)
mcad.obsm[f"5kCG_u{npc}_seuratL2_tsne"] = mcad.obsm["X_tsne"].copy()

In [20]:
mcad.write_h5ad("5kCG_lsi.h5ad")

## 4b · 3C all-cell (refined anchors)

_Source: `clustering/merged/L1/02.100k3C_clustering.ipynb`_

In [21]:
tmp = anndata.read_h5ad(f"../L1pre/100k3C_pca.h5ad")
tmp

In [22]:
mcad = anndata.AnnData(
    obs=tmp.obs[
        [
            "FinalmCReads",
            "mCHFrac",
            "mCGFrac",
            "CisLongContact",
            "Cis/Trans",
            "Donor",
            "Tissue",
            "celltype",
            "ClusterTissue",
        ]
    ]
)
mcad.obsm["100k3C_pca"] = tmp.obsm["100k3C_pca"].copy()
mcad.obsm["100k3C_pc50_tsne"] = tmp.obsm["100k3C_pc50_tsne"].copy()

In [23]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm="100k3C_pca", update=False)

In [24]:
npc = 50
mcad.obsm["X_pca"] = normalize(mcad.obsm["100k3C_pca"][:, :npc], axis=1)

In [25]:
sample_list = np.sort(mcad.obs["Donor"].astype(str).unique())
sample_list

In [26]:
adata_list = [mcad[mcad.obs["Donor"] == xx] for xx in sample_list]

In [27]:
anchor_index = {}
for xx, adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index("cell_id")["index"].to_dict()

In [28]:
ct_list = np.sort(
    [xx.split("/")[-2] for xx in glob(f"{outdir}tissue/L2/*-*/HiC_100kb_seurat_anchor_*.hdf")]
)
print(len(ct_list))

In [29]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f"{outdir}tissue/L2/{ct}/HiC_100kb_seurat_anchor_*.hdf")[0]
    adata_file = glob(f"{outdir}tissue/L2/{ct}/HiC_100kb_pca.h5ad")[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs["Donor"].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs["Donor"] == xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp["dist_pc"], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp["dist_pc"] < thres, ["x1", "x2", "score"]].copy()
    anchor_tmp["x1"] = cell_list[0][anchor_tmp["x1"]].map(anchor_index[sample_tmp[0]])
    anchor_tmp["x2"] = cell_list[1][anchor_tmp["x2"]].map(anchor_index[sample_tmp[1]])
    anchor_tmp["x1_donor"] = sample_tmp[0]
    anchor_tmp["x2_donor"] = sample_tmp[1]
    anchor.append(anchor_tmp)

anchor = pd.concat(anchor, axis=0)

In [30]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i, j]]
        selc = (anchor["x1_donor"] == x1) & (anchor["x2_donor"] == x2)
        anchor_df[(i, j)] = anchor.loc[selc, ["x1", "x2", "score"]]
        if selc.sum() == 0:
            selc = (anchor["x1_donor"] == x2) & (anchor["x2_donor"] == x1)
            anchor_df[(i, j)] = anchor.rename({"x1": "x2", "x2": "x1"}, axis=1).loc[
                selc, ["x1", "x2", "score"]
            ]

In [31]:
integrator = SeuratIntegration()

In [32]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = "X_pca"
integrator._calculate_local_knn()

integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df

In [33]:
corrected = integrator.integrate(
    key_correct="100k3C_pca", row_normalize=True, n_components=npc, k_weight=100, sd=1
)

In [34]:
corrected = pd.DataFrame(
    normalize(np.concatenate(corrected, axis=0), axis=1),
    index=np.concatenate([xx.obs.index for xx in adata_list]),
)

mcad.obsm[f"100k3C_pc{npc}_seuratL2"] = corrected.loc[mcad.obs.index].values
mcad.obsm[f"100k3C_pc{npc}_seuratL2"] = normalize(
    mcad.obsm[f"100k3C_pc{npc}_seuratL2"][:, :npc], axis=1
)

tsne(
    mcad,
    obsm=f"100k3C_pc{npc}_seuratL2",
    metric="euclidean",
    exaggeration=-1,
    perplexity=50,
    n_jobs=-1,
)
mcad.obsm[f"100k3C_pc{npc}_seuratL2_tsne"] = mcad.obsm["X_tsne"].copy()

In [35]:
mcad.write_h5ad("100k3C_pca.h5ad")

## 4c · joint embedding → final L1 + split to L2

_Source: `clustering/merged/L1/03.joint_embed.ipynb`_

In [36]:
group_name = "All"

In [37]:
tmp = anndata.read_h5ad("../L1pre/5kCG100k3C_embed.h5ad")
tmp

In [38]:
adata_mc = anndata.read_h5ad("5kCG_lsi.h5ad")
adata_3c = anndata.read_h5ad("100k3C_pca.h5ad")
adata_3c = adata_3c[adata_mc.obs.index].copy()

In [39]:
npc_cg = [int(xx.split("_")[1][1:]) for xx in adata_mc.obsm.keys() if "_seuratL2_tsne" in xx][0]
npc_3c = [int(xx.split("_")[1][2:]) for xx in adata_3c.obsm.keys() if "_seuratL2_tsne" in xx][0]
print(npc_cg, npc_3c)

In [40]:
adata = anndata.AnnData(obs=adata_mc.obs)
adata.obs["L1"] = tmp.obs["L1"].copy()
# adata.obsm[f'5kCG100k3C_u{npc_cg}u{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}hm'], axis=1),
adata.obsm[f"5kCG100k3C_u{npc_cg}pc{npc_3c}"] = np.hstack(
    [
        normalize(adata_mc.obsm[f"5kCG_u{npc_cg}_seuratL2"], axis=1),
        normalize(adata_3c.obsm[f"100k3C_pc{npc_3c}_seuratL2"], axis=1),
    ]
)
tsne(
    adata,
    obsm=f"5kCG100k3C_u{npc_cg}pc{npc_3c}",
    metric="euclidean",
    exaggeration=-1,
    perplexity=50,
    n_jobs=-1,
)
adata.obsm[f"5kCG100k3C_u{npc_cg}pc{npc_3c}_tsne"] = adata.obsm["X_tsne"].copy()

In [41]:
adata.write_h5ad("5kCG100k3C_embed.h5ad")

In [42]:
adata = anndata.read_h5ad("5kCG100k3C_embed.h5ad")

In [43]:
# clustering name
clustering_name = "L1"

# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.95
min_cluster_size = 100

# Other ConsensusClustering parameters
metric = "euclidean"
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 32

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = "correlation"
method_hclust = "average"

plot_type = "static"

In [44]:
cc = ConsensusClustering(
    model=None,
    n_neighbors=n_neighbors,
    metric=metric,
    min_cluster_size=min_cluster_size,
    leiden_repeats=leiden_repeats,
    leiden_resolution=leiden_resolution,
    consensus_rate=consensus_rate,
    random_state=random_state,
    train_frac=train_frac,
    train_max_n=train_max_n,
    max_iter=max_iter,
    n_jobs=n_jobs,
    target_accuracy=target_accuracy,
)

In [45]:
# )


# step = max(int(leiden_repeats / n_jobs), 10)

#             **partition_kwargs,
#         )

#         try:
#         except Exception as exc:
# )

In [46]:
cc.fit_predict(adata.obsm[f"5kCG100k3C_u{npc_cg}pc{npc_3c}"])

In [47]:
adata.obs["leiden_cons"] = cc.label.copy()
adata.obs["leiden_cv"] = cc.cv_predicted_label.copy()

In [48]:
cc.save("ConcensusClustering.model.lib")
adata.write_h5ad("5kCG100k3C_embed.h5ad")

In [49]:
adata = anndata.read_h5ad("5kCG100k3C_embed.h5ad")

In [50]:
indir = f"{ENTEX_ROOT}/"
tissue_list = np.sort([xx.split("/")[-2] for xx in glob(f"{indir}tissue/*/")])
tissue_list

In [51]:
leg = [
    "Hema Myeloid",
    "Hema Mast",
    "Hema B",
    "Hema Tnaive",
    "Hema Tmem",
    "Hema NK",
    "Glia Astro",
    "Glia Oligo",
    "Neu Exc",
    "Neu Inh",
    "Neu Schw",
    "Epi Endcri",
    "Epi Duc",
    "Epi Aci",
    "Epi Krt/Lum",
    "Epi Alv",
    "Epi TPB",
    "Epi Gas",
    "Epi AdrCtx",
    "Epi Ent",
    "Epi BrstBasal",
    "Mus Skl",
    "Mus Crd",
    "Endo Lym",
    "Endo Ves",
    "SmMus/Peri",
    "Fibro HT",
    "Fibro GI",
    "Fibro B",
    "Fibro EndN",
    "Fibro EpiN",
    "Fibro Sk",
    "Fibro Mus",
    "Fibro Adr",
    "Fibro Myo",
]

In [52]:
selc = [
    [48, 46, 2, 35, 53, 22, 56],
    [25],
    [37, 30, 16],
    [12],
    [0, 8],
    [29],
    [40],
    [15, 59],
    [13, 42, 54],
    [33, 36],
    [20],
    [11],
    [39],
    [18],
    [47, 44, 41],
    [17],
    [1, 51],
    [4, 34],
    [5],
    [50, 7],
    [26],
    [3, 62],
    [27],
    [45],
    [10, 58, 49, 55, 63, 21],
    [32, 24],
    [6],
    [28, 57, 9, 61],
    [14],
    [19, 60],
    [31],
    [23],
    [43],
    [38, 64],
    [52],
]

In [53]:
clusterdict = {f"c{xx}": yy for x, yy in zip(selc, leg) for xx in x}

adata.obs["L1"] = adata.obs["leiden_cons"].map(clusterdict).astype(str)
adata.obs["L1"].value_counts()

In [54]:
adata.write_h5ad("5kCG100k3C_embed.h5ad")

In [55]:
sample_list = np.sort(adata.obs["Donor"].astype(str).unique())
sample_list

In [56]:
outdir = f"{ENTEX_ROOT}/clustering/merged/"

In [57]:
mcad = anndata.read_h5ad(f"{outdir}5kCG.h5ad")
mcad.obs[["celltype", "ClusterTissue"]] = adata.obs.loc[
    mcad.obs.index, ["L1", "ClusterTissue"]
].values

In [58]:
for xx in mcad.obs["celltype"].unique():
    tmp = mcad[mcad.obs["celltype"] == xx]
    x = xx.replace(" ", "-").replace("/", "_")
    os.makedirs(f"{outdir}L2/{x}/", exist_ok=True)
    tmp.write_h5ad(f"{outdir}L2/{x}/5kCG.h5ad")

In [59]:
indir = f"{ENTEX_ROOT}/tissue/"
file_list = glob(f"{indir}*/data/cell_table.tsv")
cell_list = pd.concat(
    [pd.read_csv(file, sep="\t", index_col=0, header=None) for file in file_list], axis=0
)

In [60]:
chromsize = pd.read_csv(
    f"{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes", sep="\t", index_col=0, header=None
)
chromsize = chromsize.iloc[:22]

In [61]:
for chrom in chromsize.index:
    file_list = glob(f"{indir}*/data/raw/{chrom}.npz")
    adata = []
    for file in file_list:
        adata.append(np.load(file)["arr_0"])
    adata = np.concatenate(adata, axis=0)
    adata = anndata.AnnData(X=adata, obs=mcad.obs.loc[cell_list.index])[mcad.obs.index]
    for xx in adata.obs["celltype"].unique():
        tmp = adata[adata.obs["celltype"] == xx]
        x = xx.replace(" ", "-").replace("/", "_")
        os.makedirs(f"{outdir}L2/{x}/raw/", exist_ok=True)
        np.savez(f"{outdir}L2/{x}/raw/{chrom}.npz", tmp.X)
    print(chrom)